# 05-05 RunnableParallel: 여러 체인을 동시에 실행하기

LLM API 호출은 네트워크 요청이기 때문에 응답을 기다리는 시간이 있어요. 체인 A가 끝나길 기다렸다가 체인 B를 실행하면 전체 대기 시간이 합산되지만, 두 체인을 동시에 실행하면 가장 느린 체인의 시간만큼만 기다리면 돼요. 이것이 `RunnableParallel`의 핵심 가치예요.

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `RunnableParallel`로 여러 체인을 동시에 실행하고 결과를 딕셔너리로 합치는 방법을 구현할 수 있어요
- 딕셔너리 형태의 자동 래핑(Auto-wrapping) 원리를 이해하고, 세 가지 동등한 표기법을 설명할 수 있어요
- `itemgetter`를 활용하여 동일한 입력 딕셔너리에서 서로 다른 키를 추출하는 패턴을 적용할 수 있어요
- 순차 실행과 병렬 실행의 처리 시간 차이를 측정하고 설명할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- `RunnablePassthrough`와 LCEL 파이프 연산자(`|`) 사용법 (Ch05-01)
- `operator.itemgetter`의 기본 사용법
- 파이썬의 멀티스레딩(Multi-threading) 개념 (심화 이해에 도움)

---

> 🔑 **핵심 개념**: `RunnableParallel`은 독립적인 체인들을 동시에 실행해 응답 시간을 줄여요. 순차 실행은 `시간 = A + B + C`이지만, 병렬 실행은 `시간 = max(A, B, C)`예요. 체인 수가 많을수록 절감 효과가 커져요.

아래 다이어그램은 병렬 실행의 흐름을 보여줘요:

```mermaid
flowchart TD
    IN["입력 딕셔너리"]:::input
    FORK["RunnableParallel<br/>분기 시작"]:::process
    C1["capital_chain<br/>수도 조회"]:::process
    C2["area_chain<br/>면적 조회"]:::process
    JOIN["결과 병합<br/>{capital: ..., area: ...}"]:::process
    OUT["최종 출력 딕셔너리"]:::output

    IN --> FORK
    FORK --> C1
    FORK --> C2
    C1 --> JOIN
    C2 --> JOIN
    JOIN --> OUT

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

**주요 특징:**

- 여러 작업을 동시에 실행하여 전체 소요 시간 단축
- 각 Runnable의 결과를 키로 구분하여 딕셔너리로 반환
- LCEL에서 `{...}` 딕셔너리를 쓰면 자동으로 `RunnableParallel`로 변환

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

## 1. RunnableParallel 기본 사용법

`RunnableParallel`은 딕셔너리 형태로 여러 Runnable을 병렬로 실행해요. 각 키에 할당된 체인이 동시에 실행되고, 결과가 같은 키 이름으로 딕셔너리에 담겨 반환돼요.

**자동 래핑(Auto-wrapping):** LCEL 파이프라인에서 딕셔너리 `{...}`를 사용하면 자동으로 `RunnableParallel`로 변환돼요. 아래 세 가지 표기법은 모두 동일하게 동작해요:

```python
# 방법 1: 딕셔너리 자동 래핑 (가장 간결, 실무에서 주로 사용)
{"capital": capital_chain, "area": area_chain}

# 방법 2: 딕셔너리를 명시적으로 전달
RunnableParallel({"capital": capital_chain, "area": area_chain})

# 방법 3: 키워드 인자 사용
RunnableParallel(capital=capital_chain, area=area_chain)
```

> 🎯 **강의 포인트**: 이전 노트북(04-Routing)에서 `{"topic": classification_chain, "question": itemgetter("question")}`을 이미 사용했어요. 그때 딕셔너리가 자동으로 `RunnableParallel`로 변환되어 `classification_chain`과 `itemgetter`가 병렬로 실행됐던 거예요. 이미 사용해본 패턴을 이번에 명시적으로 배우는 셈이에요.

> ⚠️ **주의**: 병렬로 실행하려는 체인들 사이에 의존 관계가 있으면 안 돼요. A의 결과가 B에 필요한 경우는 순차 실행(`|`)을 사용해야 해요. 예를 들어 "수도를 먼저 조회하고, 그 수도의 인구를 조회"하는 경우는 병렬이 아닌 순차 파이프라인으로 연결해야 해요.

In [3]:
# ============================================================
# TODO: RunnableParallel로 두 체인을 병렬 실행하는 코드를 완성하세요
# 힌트: RunnableParallel(capital=capital_chain, area=area_chain)
# 예상 결과: {'capital': '대한민국의 수도는 서울입니다.', 'area': '약 100,210 km²...'}
# ============================================================

# ---------------------------------------------------
# RunnableParallel 기본 예제 — 병렬로 두 체인 실행하기
# ---------------------------------------------------

# 1단계: 수도 정보를 조회하는 체인
capital_chain = (
    ChatPromptTemplate.from_template("{country}의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

# 2단계: 면적 정보를 조회하는 체인
area_chain = (
    ChatPromptTemplate.from_template("{country}의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)

# 3단계: RunnableParallel로 병렬 실행
# 세 가지 표기법은 모두 동일:
#   RunnableParallel(capital=capital_chain, area=area_chain)
#   RunnableParallel({"capital": capital_chain, "area": area_chain})
#   {"capital": capital_chain, "area": area_chain}  ← LCEL에서 자동 래핑
parallel_chain = RunnableParallel(
    capital=capital_chain,
    area=area_chain
)

# 4단계: 실행 결과 확인
result = parallel_chain.invoke({"country": "대한민국"})

print("=" * 60)
print("📥 병렬 실행 결과")
print("=" * 60)
print(f"입력: {{'country': '대한민국'}}")
print()
print("📤 출력:")
for key, value in result.items():
    print(f"  {key}: {value}")
print()
print("💡 두 체인이 병렬로 실행되어 거의 동일한 시간에 완료됨")


📥 병렬 실행 결과
입력: {'country': '대한민국'}

📤 출력:
  capital: 대한민국의 수도는 서울입니다.
  area: 대한민국의 면적은 약 100,210 평방킬로미터(약 38,691 평방 마일)입니다. 이는 한반도의 남쪽에 위치한 지역으로, 여러 산과 강, 해안선이 있는 다양한 지형을 포함하고 있습니다.

💡 두 체인이 병렬로 실행되어 거의 동일한 시간에 완료됨


## 2. itemgetter를 활용한 RAG 예제

`operator.itemgetter`는 파이썬 표준 라이브러리의 함수로, 딕셔너리에서 특정 키의 값을 추출해요. `itemgetter("question")`은 `lambda x: x["question"]`과 동일하지만, LCEL에서 Runnable로 자동 변환되기 때문에 파이프라인 안에서 바로 사용할 수 있어요.

`RunnableParallel` 안에서 입력 딕셔너리의 서로 다른 키를 각각의 체인에 전달할 때 자주 사용해요. 특히 RAG 패턴에서는 `question` 키로 검색하고, `language` 키로 언어를 지정하는 식으로 하나의 입력에서 여러 값을 분리해 전달하는 구조가 일반적이에요.

> 💡 **실무 팁**: RAG 체인에서 `itemgetter`는 거의 필수 도구예요. `context`는 검색 결과를, `question`은 원본 질문을 전달하는 패턴이 RAG의 표준 구조가 돼요. 이 패턴은 Ch06 RAG 챕터에서 반복적으로 사용하게 돼요.

In [4]:
# ============================================================
# TODO: itemgetter를 활용하여 입력 딕셔너리에서 값을 추출하는 RAG 체인을 완성하세요
# 힌트: itemgetter("question") — 입력 딕셔너리의 "question" 키 값 추출
# 예상 결과: "파이썬은 데이터 과학과 머신러닝 분야에서 널리 사용됩니다."
# ============================================================

# ---------------------------------------------------
# itemgetter + RunnableParallel — RAG 패턴 구현
# ---------------------------------------------------

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# 1단계: 벡터 저장소 생성 (간단한 예시용 데이터)
vectorstore = Chroma.from_texts(
    ["파이썬은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다."],
    embedding=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()

# 문서 리스트를 하나의 문자열로 합치는 헬퍼 함수
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

# RAG 프롬프트 템플릿
template = """다음 컨텍스트를 바탕으로 질문에 답변해주세요.
{language}로 답변해주세요.

컨텍스트:
{context}

질문: {question}

답변:"""

prompt = ChatPromptTemplate.from_template(template)

# 2단계: itemgetter로 입력 딕셔너리에서 필요한 값 추출
# 이 딕셔너리 부분이 RunnableParallel로 자동 변환됨:
#   - context: question으로 벡터 검색 → 문서 포맷팅 (체인 연결)
#   - question: 입력에서 그대로 추출
#   - language: 입력에서 그대로 추출
rag_chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)

# 3단계: 실행
result = rag_chain.invoke({
    "question": "파이썬은 어떤 분야에서 사용되나요?",
    "language": "한국어"
})

print("=" * 60)
print("📥 RAG 체인 실행 결과")
print("=" * 60)
print(result)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Number of requested results 4 is greater than number of elements in index 1, updating n_results = 1


📥 RAG 체인 실행 결과
파이썬은 데이터 과학과 머신러닝을 비롯하여 웹 개발, 자동화, 데이터 분석, 인공지능, 게임 개발 등 다양한 분야에서 널리 사용됩니다.


## 3. 서로 다른 입력 변수를 사용하는 예제

`RunnableParallel` 안의 각 체인이 서로 다른 입력 변수를 사용할 수도 있어요. 예를 들어 한 체인은 `country1` 키를, 다른 체인은 `country2` 키를 사용할 수 있어요.

이 경우 `invoke()` 호출 시 모든 체인이 필요로 하는 키를 모두 포함한 딕셔너리를 전달하면 돼요. 각 체인은 자신에게 필요한 키만 사용하고 나머지는 무시해요.

> 🔑 **핵심 개념**: `RunnableParallel`은 입력 딕셔너리 전체를 모든 체인에 전달해요. 각 체인의 프롬프트 템플릿이 필요한 변수만 골라서 사용하기 때문에, 서로 다른 변수명을 써도 문제없이 동작해요.

In [5]:
# ============================================================
# TODO: 서로 다른 입력 변수(country1, country2)를 사용하는 병렬 체인을 완성하세요
# 힌트: 각 체인이 다른 키를 사용하더라도 invoke()에 모든 키를 포함하면 됨
# 예상 결과: {'capital': '서울', 'area': '미국의 면적은 약 9,830,000 km²'}
# ============================================================

# ---------------------------------------------------
# 서로 다른 입력 변수를 사용하는 병렬 체인
# ---------------------------------------------------

# 1단계: 각기 다른 변수명을 사용하는 체인 정의
# - capital_chain_1: {country1} 변수를 사용
# - area_chain_2: {country2} 변수를 사용
# 같은 RunnableParallel 안이지만 각 체인이 다른 키를 참조함
capital_chain_1 = (
    ChatPromptTemplate.from_template("{country1}의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

area_chain_2 = (
    ChatPromptTemplate.from_template("{country2}의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)

# 2단계: 병렬 실행 (서로 다른 입력 변수 사용)
# RunnableParallel이 입력 딕셔너리 전체를 두 체인에 모두 전달함
# capital_chain_1은 country1만, area_chain_2는 country2만 사용
parallel_chain = RunnableParallel(
    capital=capital_chain_1,
    area=area_chain_2
)

# 3단계: 실행 — 두 체인에 필요한 키를 모두 제공
result = parallel_chain.invoke({
    "country1": "대한민국",
    "country2": "미국"
})

print("=" * 60)
print("📥 서로 다른 입력 변수 사용 예제")
print("=" * 60)
print(f"입력: {{'country1': '대한민국', 'country2': '미국'}}")
print()
print("📤 출력:")
for key, value in result.items():
    print(f"  {key}: {value}")

📥 서로 다른 입력 변수 사용 예제
입력: {'country1': '대한민국', 'country2': '미국'}

📤 출력:
  capital: 대한민국의 수도는 서울입니다.
  area: 미국의 면적은 약 9,830,000 평방킬로미터(3,796,000 평방마일)입니다. 이는 미국이 세계에서 세 번째로 큰 국가임을 의미합니다. 러시아에 이어 두 번째로 큰 국가는 캐나다입니다.


## 4. 병렬 실행 성능 비교

`RunnableParallel`을 사용하면 여러 작업을 병렬로 실행하여 전체 실행 시간을 단축할 수 있어요. 이 효과는 LLM API 호출처럼 I/O 대기 시간이 긴 작업에서 특히 두드러져요.

순차 실행과 병렬 실행의 시간 차이를 직접 측정해볼게요.

```
순차 실행: ████████ (chain A) ████████ (chain B)  → 총 시간 = A + B
병렬 실행: ████████ (chain A)                      → 총 시간 = max(A, B)
           ████████ (chain B)
```

> 💡 **실무 팁**: LLM API 호출처럼 I/O 대기가 긴 작업에서 병렬 실행의 효과가 커요. CPU 연산 위주의 작업에서는 파이썬의 GIL(Global Interpreter Lock) 때문에 병렬 효과가 제한적이지만, 네트워크 요청은 GIL의 영향을 받지 않아서 체인 수에 비례한 속도 향상을 기대할 수 있어요.

> 🎯 **강의 포인트**: 실제 API 호출이 포함된 성능 비교 실험은 학생들에게 병렬화의 실질적 효과를 체감하게 해요. 이론으로만 설명하는 것보다 숫자로 보여주는 것이 효과적이에요. 체인 수를 3~4개로 늘려 테스트하면 차이가 더 극적으로 나타나요.

In [6]:
import time

# ============================================================
# TODO: sequential_execution()과 parallel_execution() 함수를 구현하고 성능을 비교하세요
# 힌트: time.time()으로 실행 전후 시간을 측정
# 예상 결과: 병렬 실행이 순차 실행보다 빠름 (1.x배 향상)
# ============================================================

# ---------------------------------------------------
# 성능 비교: 순차 실행 vs 병렬 실행
# ---------------------------------------------------

# 공통 시스템 프롬프트 — 두 체인이 동일한 역할 설정을 공유
shared_instruction = (
    "당신은 세계 지리에 정통한 조교입니다. "
    "항상 신뢰할 수 있는 최신 정보를 한국어로 한 문장으로 전달하세요."
)

# 1단계: 수도 정보를 조회하는 체인
capital_prompt = ChatPromptTemplate.from_messages([
    ("system", shared_instruction),
    (
        "human",
        (
            "국가 이름: {country}\n"
            "요청: 해당 국가의 행정수도 한 곳만 답해주세요."
        ),
    ),
])
capital_chain = (
    capital_prompt
    | model
    | StrOutputParser()
)

# 2단계: 면적 정보를 조회하는 체인
area_prompt = ChatPromptTemplate.from_messages([
    ("system", shared_instruction),
    (
        "human",
        (
            "국가 이름: {country}\n"
            "요청: 대표 면적(km^2 단위)을 수치와 단문 설명으로 알려주세요."
        ),
    ),
])
area_chain = (
    area_prompt
    | model
    | StrOutputParser()
)

# 3단계: RunnableParallel로 병렬 체인 구성
parallel_chain = RunnableParallel(
    capital=capital_chain,
    area=area_chain,
)

# 4단계: 순차 실행 — 각 체인을 하나씩 순서대로 호출
# 총 시간 = capital_chain 시간 + area_chain 시간
def sequential_execution():
    """두 체인을 순차적으로 실행"""
    start_time = time.time()
    
    result1 = capital_chain.invoke({"country": "대한민국"})
    result2 = area_chain.invoke({"country": "대한민국"})
    
    sequential_time = time.time() - start_time
    return {"capital": result1, "area": result2}, sequential_time

# 5단계: 병렬 실행 — RunnableParallel로 두 체인을 동시에 실행
# 총 시간 = max(capital_chain 시간, area_chain 시간)
def parallel_execution():
    """두 체인을 병렬로 실행"""
    start_time = time.time()
    
    result = parallel_chain.invoke({"country": "대한민국"})
    
    parallel_time = time.time() - start_time
    return result, parallel_time

# 6단계: 성능 비교 실행
print("=" * 60)
print("⚡ 프롬프트 재정의 후 성능 비교")
print("=" * 60)

seq_result, seq_time = sequential_execution()
par_result, par_time = parallel_execution()

print("📤 순차 실행 결과:")
for key, value in seq_result.items():
    print(f"  {key}: {value}")
print()
print("📤 병렬 실행 결과:")
for key, value in par_result.items():
    print(f"  {key}: {value}")
print()
print(f"순차 실행 시간: {seq_time:.2f}초")
print(f"병렬 실행 시간: {par_time:.2f}초")
print()
if par_time > 0:
    print(f"성능 향상: {seq_time / par_time:.2f}배 빠름")
else:
    print("성능 향상: 병렬 실행 시간이 0에 가까워 계산 불가")
print()
print("💡 핵심 정리")
print("=" * 60)
print("• RunnableParallel은 재정의된 프롬프트를 공유하며 독립 작업을 병렬로 실행")
print("• 순차 실행 대비 실행 시간이 크게 단축됨")
print("• 각 체인의 결과는 딕셔너리 키로 구분되어 반환됨")

⚡ 프롬프트 재정의 후 성능 비교
📤 순차 실행 결과:
  capital: 대한민국의 행정수도는 세종특별자치시입니다.
  area: 대한민국의 대표 면적은 약 100,210㎢로, 한반도의 남쪽에 위치한 국가로서 다양한 지형과 기후를 가지고 있습니다.

📤 병렬 실행 결과:
  capital: 대한민국의 행정수도는 세종시입니다.
  area: 대한민국의 면적은 약 100,210 km²로, 한반도의 남부에 위치한 국가입니다.

순차 실행 시간: 3.78초
병렬 실행 시간: 1.08초

성능 향상: 3.49배 빠름

💡 핵심 정리
• RunnableParallel은 재정의된 프롬프트를 공유하며 독립 작업을 병렬로 실행
• 순차 실행 대비 실행 시간이 크게 단축됨
• 각 체인의 결과는 딕셔너리 키로 구분되어 반환됨


## 정리

### RunnableParallel 주요 메서드 및 패턴

| 패턴 | 코드 예시 | 설명 |
|------|----------|------|
| 딕셔너리 자동 래핑 | `{"a": chain_a, "b": chain_b}` | LCEL에서 가장 흔히 쓰이는 형태 |
| 명시적 생성 | `RunnableParallel(a=chain_a, b=chain_b)` | 의도를 명확히 드러내고 싶을 때 |
| itemgetter 조합 | `{"q": itemgetter("question")}` | 입력 딕셔너리에서 특정 키 추출 |
| 체인 파이프 조합 | `{"ctx": itemgetter("q") \| retriever}` | 추출 후 추가 처리까지 연결 |

### 병렬 vs 순차 실행 판단 기준

| 조건 | 실행 방식 | 이유 |
|------|----------|------|
| 체인 A, B가 독립적 | `RunnableParallel` (병렬) | 동시 실행으로 시간 단축 |
| B가 A의 결과를 필요로 함 | `A \| B` (순차) | 의존성이 있어 순서가 필요 |
| 입력 딕셔너리에서 여러 키 추출 | `RunnableParallel` + `itemgetter` | 각 키를 별도 체인에 전달 |

### 핵심 요점

- `RunnableParallel`은 독립적인 여러 체인을 동시에 실행하여 전체 소요 시간을 줄여줘요
- 딕셔너리 형태 `{key: runnable, ...}`로 작성하면 자동으로 병렬 실행이 적용돼요
- `itemgetter`로 입력 딕셔너리에서 필요한 키만 추출해 각 체인에 전달할 수 있어요
- 서로 다른 입력 변수를 사용하는 체인도 하나의 `RunnableParallel`에 묶을 수 있어요
- I/O 대기가 긴 작업(LLM API 호출)에서 병렬 실행의 성능 이점이 두드러져요

### 다음 노트북 예고

다음 노트북(`06-Configure.ipynb`)에서는 체인을 실행할 때 런타임(Runtime)에 동적으로 설정을 변경하는 방법을 배워요. 모델이나 프롬프트를 호출 시점에 교체하는 `configurable_fields`와 `configurable_alternatives`를 살펴볼게요.